# BGE-M3 Embedding — GDPRScope chunks

**Objetivo:** Embeber ~84K chunks con BGE-M3 (1024 dims, multilingue) en Colab T4 (gratis).

**Flujo:**
1. Subir `chunks_to_embed.csv.gz` (exportado de la DB)
2. Cargar BGE-M3 en GPU
3. Generar embeddings en batches
4. Guardar `.npy` → descargar → importar en DB

**Runtime:** Cambiar a GPU → Runtime > Change runtime type > T4 GPU

In [ ]:
# Cell 1: Install dependencies
!pip install -q sentence-transformers pandas pyarrow

In [ ]:
# Cell 2: Upload chunks CSV (compressed)
from google.colab import files
import pandas as pd

uploaded = files.upload()  # Upload chunks_to_embed.csv.gz
filename = list(uploaded.keys())[0]

if filename.endswith('.gz'):
    df = pd.read_csv(filename, compression='gzip')
else:
    df = pd.read_csv(filename)

print(f'Loaded {len(df)} chunks')
print(f'Sections: {df["section"].value_counts().to_dict()}')
print(f'Avg content length: {df["content"].str.len().mean():.0f} chars')
print(f'Max content length: {df["content"].str.len().max()} chars')

In [ ]:
# Cell 3: Load BGE-M3 model on GPU
import torch
from sentence_transformers import SentenceTransformer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

model = SentenceTransformer('BAAI/bge-m3', device=device)
print(f'Model loaded: {model.get_sentence_embedding_dimension()} dims')

In [ ]:
# Cell 4: Generate embeddings in batches
import numpy as np
import time

texts = df['content'].tolist()
BATCH_SIZE = 64
all_embeddings = []

start = time.time()
for i in range(0, len(texts), BATCH_SIZE):
    batch = texts[i:i + BATCH_SIZE]
    embs = model.encode(batch, normalize_embeddings=True, show_progress_bar=False)
    all_embeddings.append(embs)
    if (i // BATCH_SIZE) % 50 == 0:
        elapsed = time.time() - start
        done = i + len(batch)
        rate = done / elapsed if elapsed > 0 else 0
        eta = (len(texts) - done) / rate if rate > 0 else 0
        print(f'  {done}/{len(texts)} ({100*done/len(texts):.1f}%) — {rate:.0f} chunks/s — ETA {eta/60:.1f} min')

embeddings = np.vstack(all_embeddings)
elapsed = time.time() - start
print(f'\nDone: {len(embeddings)} embeddings in {elapsed/60:.1f} min')
print(f'Shape: {embeddings.shape}')
print(f'Rate: {len(embeddings)/elapsed:.0f} chunks/s')

In [ ]:
# Cell 5: Save results
np.save('embeddings_bge_m3.npy', embeddings)
np.save('chunk_ids.npy', df['chunk_id'].values)
print(f'Saved embeddings_bge_m3.npy ({embeddings.shape})')
print(f'Saved chunk_ids.npy ({len(df)} ids)')

In [ ]:
# Cell 6: Download results
files.download('embeddings_bge_m3.npy')
files.download('chunk_ids.npy')